# Read in the data

In deze notebook willen we laten zien hoe je synthpop-py gebruikt.
Daarnaast willen we in dit notebook ook laten zien hoe synthpop-py business rules nu niet altijd meeneemt.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('Synthetic_Homicide.csv')

df.head(5)

# Synthesise the dataset

In [ ]:
# """
# Installeer synthpop-py. Zorg dat je na installatie de Jupyter kernel herstart.
# Bekijk eventueel ook: https://synthpop-py.readthedocs.io/en/develop/getting_started/installation.html
# """
# !pip install synthpop-py

In [ ]:
from synthpop import Synthesiser

syn_df = Synthesiser(random_seed=2834727).fit(df).generate(50000)#We maken veel meer synthetische rijen dan dat er originele rijen zijn, zodat de gebreken goed naarvoren komen.
syn_df.head(5)

# Business rules check

Hier laten we zien dat de business rules niet worden behouden tijdens het synthese process. Zie het document voor meer achtergrond informatie

In [ ]:
"""mapping tussen geboorteland en geboorteplaats:"""

city_mapping = {
        'Netherlands': [
            'Amsterdam', 'Rotterdam', 'The Hague', 'Utrecht', 'Eindhoven',
            'Groningen', 'Tilburg', 'Almere', 'Breda', 'Nijmegen',
            'Harlingen', #in Friesland
            'Napels', #groningen
            'Leuven', #Gelderland
            'Zurich', #Friesland
            'Spitsbergen', #Groningen
            'Deurne',# Noord-Brabandt
            'Wezel', #Wijchen, Gelderland
        ],
        'Other non-European': [
            'New York', 'Tokyo', 'Dubai', 'Sydney', 'Cape Town',
            'Hong Kong', 'Bangkok', 'Kuala Lumpur', 'Jakarta', 'Lagos',
            'Harlingen',# Amerika
            'Bristol',#canada
            'London',#Amerika
            'Paris', #Texas, Amerika
            'Napels',#Florida, Amerika (eigenlijk Naples)
            'Barcelona',#Venezuela
        ],
        'Other Europe': [
            'Berlin', 'Paris', 'Madrid', 'Rome', 'Vienna',
            'Prague', 'Brussels', 'Warsaw', 'Budapest', 'Lisbon',
            'Athens', 'Copenhagen', 'Stockholm', 'Oslo', 'Helsinki',
            'Dublin', 'Zurich', 'Geneva', 'Munich', 'Hamburg',
            'Napels','Leuven','Zurich','Spitsbergen',
            'Bristol',#VK
            'London',#VK
            'Barcelona',
            'Deurne', #België
            'Wezel', #België
        ],
        np.nan: [
            np.nan
        ]
    }

In [ ]:
"""1. Geboorteplaats en Geboorteland"""

def check_city_country(row):
    #return True als de stad overeenkomt met het land
    city_perp = row['Birthcity of main perpetrator'] 
    country_perp = row['Birthcountry of main perpetrator']

    country_not_na = pd.notna(country_perp)
    country_is_na = pd.isna(country_perp)
    city_na = pd.isna(city_perp)

    
    if (not city_na) and country_is_na:
        return False #Als de stad bekend is, moet het land ook bekend zijn.

    if city_na and country_is_na:
        return True #stad en land mogen beide onbekend zijn.

    if country_not_na and city_na:
        return True #Het is mogelijk dat het land wel bekend is, maar de stad niet
        
    city_matches = city_perp in city_mapping[country_perp]

    return city_matches# if country_not_na else city_na

mask_city_mapping = ~syn_df.apply(
    check_city_country,
        axis=1
    )

invalid_rows = syn_df[mask_city_mapping]
invalid_combinations = (
    invalid_rows[['Birthcountry of main perpetrator', 'Birthcity of main perpetrator']]
    .drop_duplicates()
)
print(
    f'Combinaties die nieuw zijn voor de geboorteplaats en geboorteland')
print(invalid_combinations)

# mask_city_mapping moet volledig False zijn als de business rules zijn behouden
assert not mask_city_mapping.any()

In [ ]:
"""2. Leeftijdsverschil"""

Wrong_age_diff = ((syn_df['Age difference'] != (
    np.abs(syn_df['Age of main perpetrator'] - syn_df['Age of victim']))) &
    (~pd.isna(syn_df['Age difference'])))  # no nans

print('Bij het juist volgen van de business rules moet de volgende dataframe leeg zijn. Immers is dan Leeftijdsverschil = leeftijd_dader - leeftijd_slachtoffer')
print(
    syn_df[['Age of main perpetrator', 'Age of victim',
            'Age difference']][Wrong_age_diff]
)

# Wrong_age_diff moet volledig False zijn als de business rules zijn behouden
assert not Wrong_age_diff.any()

In [ ]:
"""3. Behoud van tijdsperioden"""

mask_crime_syn = ((syn_df['Date of previous crime main perpetrator'] > syn_df['Date of current crime']) | (
    syn_df['Date of current crime'] > syn_df['Date of release from current crime main perpetrator']))
print('Momenten van tijd die elkaar incorrect opvolgen')
print(
    syn_df[['Date of previous crime main perpetrator', 'Date of current crime',
            'Date of release from current crime main perpetrator']][mask_crime_syn]
)

# mask_crime_syn moet volledig False zijn als de business rules zijn behouden
assert not mask_crime_syn.any()

In [ ]:
"""4. Maximale gevangenisstraf"""
mask_prison_sentence_syn = (
    (
        (syn_df['Solved vs unsolved homicide'] == 'Solved') &
        (syn_df['Years prison sentence main perpetrator'] > 10.)
    ) |
    (
        (syn_df['Solved vs unsolved homicide'] == 'Unsolved') &
        (syn_df['Years prison sentence main perpetrator'].notna())
    )
)

print('Zou leeg moeten zijn volgens de Business rules')

print(
    syn_df[['Solved vs unsolved homicide',
            'Years prison sentence main perpetrator']][mask_prison_sentence_syn]
    .value_counts(dropna=False)
)

# mask_prison_sentence_syn moet volledig False zijn als de business rules zijn behouden
assert not mask_prison_sentence_syn.any()

In [ ]:
"""5. Duur van gevangenisstraf"""
mask_age_18_syn = (
    (syn_df['Age of main perpetrator'] < 18.) &
    (syn_df['Years prison sentence main perpetrator'] > 5.) &
    ~(
        (syn_df['Age of main perpetrator'] > 12.) &
        (syn_df['Victim-perpetrator relationship'].isin(['(Step)Parent', 'Other relative']))
    )
)
print('Zou leeg moeten zijn volgens de Business rules')

print(
    syn_df[['Age of main perpetrator', 'Years prison sentence main perpetrator',
            'Victim-perpetrator relationship']][mask_age_18_syn].value_counts(dropna=False)
)

# mask_age_18_syn moet volledig False zijn als de business rules zijn behouden
assert not mask_age_18_syn.any()

In [ ]:
"""6. Business rule m.b.t. de 'Main Evidence Type'"""

"""6.1. Leeg als de zaak onopgelost is"""
mask_unsolved_syn = (syn_df['Solved vs unsolved homicide'] == 'Unsolved')

print("Check voor gevallen waarbij de zaak onopgelost is:")
print(syn_df['Main evidence type']
      [mask_unsolved_syn].value_counts(dropna=False))

# Alles moet nan zijn in dit geval
assert pd.isna(syn_df['Main evidence type'][mask_unsolved_syn]).all()

In [ ]:
"""6.2. Waarde 'bullets' en 'ballistics' zijn alleen toegestaan als het moordwapen een vuurwapen is"""
mask_firearm_syn = (
    ~mask_unsolved_syn &
    (syn_df['Main evidence type'].isin(['bullets', 'ballistics']))
)

print("Opgeloste zaken waarbij het hoofdwapen een vuurwapen is. In dit geval is elke mogelijke waarde toegestaan.")
print("Main evidence type distribution (should be: bullets, ballistics, DNA, witnesses, fingerprints, confession):")
print(syn_df['Type of weapon used']
      [mask_firearm_syn].value_counts(dropna=False))


assert (syn_df['Type of weapon used'][mask_firearm_syn] == 'Firearm').all()


In [ ]:
"""6.3 De waarde confession is niet toegestaan als de dader ontoerekeningsvatbaar is. """
mask_no_mentally_disturb = (~mask_unsolved_syn &
    (syn_df['Context of homicide'] == "Killing by mentally disturbed person (non-family)")
)

print("Gevallen waarbij de hoodsverdachte ontoerekeningsvatbaar is.")
print("'confession' is daarbij niet toegestaan")
print(syn_df['Main evidence type']
      [mask_no_mentally_disturb].value_counts(dropna=False))

assert not (syn_df['Main evidence type']
            [mask_no_mentally_disturb] == 'confession').any()

In [ ]:
"""6.4. De waarde confession is enkel toegestaan als alles is ingevuld."""
# Check for cases where all_cols are not all NaN
all_cols = [
    'Urban vs rural', 'Time of day', 'Single vs multiple homicide victims',
    'Single vs multiple homicide perpetrators', 'Type of crimescene',
    'Type of weapon used', 'Context of homicide',
    'Victim-perpetrator relationship', 'Solved vs unsolved homicide',
    'Gender of victim', 'Gender of main perpetrator', 'Age of victim',
    'Age of main perpetrator', 'Birthcountry of victim',
    'Birthcountry of main perpetrator', 'Birthcity of main perpetrator',
    'Age difference', 'Date of current crime'
]

mask_any_nan_syn =  (~mask_unsolved_syn & 
    (syn_df[all_cols].isna().any(axis=1))
    ) 

print("Confession is enkel toegestaan als deze kolommen niet NaN zijn")
print(syn_df['Main evidence type']
      [mask_any_nan_syn].value_counts(dropna=False))


assert not (syn_df['Main evidence type']
            [mask_any_nan_syn] == 'confession').any()


In [ ]:
"""6.5. De waardes DNA, fingerprints en witness zijn altijd toegestaan"""
mask_valid_values = pd.Series(['DNA', 'fingerprints', 'witnesses']).isin(syn_df['Main evidence type'])


print("'DNA', 'fingerprints', 'witnesses' zijn altijd toegestaan en komen logischerwijs voor")
print(syn_df['Main evidence type'].value_counts())

assert mask_valid_values.all()